# Torghut execution evidence

**TL;DR.** This notebook shows 30 days of TCA evidence and up to 180 days of server-aggregated runtime
ledger history. P&L is always split by `observed_stage` and `account_label`. Historical paper/replay
evidence is not broker equity. If no current live ledger exists, the notebook renders
**CURRENT PROFITABILITY UNPROVEN** instead of a zero-P&L line.

## Context and method

Slippage, expected/realized shortfall, divergence, and fill coverage come only from `execution_tca_metrics`. No order payload or decision JSON is selected.

In [ ]:
import json
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

from app.notebook_data import (
    Window,
    adapter_from_environment,
    capital_authority,
    execution_evidence,
    flow_snapshot,
    mode_banner,
    strategy_lifecycle,
)

adapter = globals().get('adapter') or adapter_from_environment()
banner = mode_banner(adapter)
banner_color = '#9a6700' if adapter.mode == 'fixture' else '#b42318'
display(HTML(f'''<div style="padding:14px 18px;border:2px solid {banner_color};border-radius:8px;
  font-weight:700;background:#fff8e6;margin-bottom:14px">{banner}</div>'''))

def bounded_points(frame, series_column, *, time_column='event_ts', preferred=5000):
    # Deterministically thin display points while preserving each series.
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    series_count = max(1, frame[series_column].nunique())
    per_series = max(2, preferred // series_count)
    chunks = []
    for _, group in frame.sort_values(time_column).groupby(series_column, sort=True):
        stride = max(1, math.ceil(len(group) / per_series))
        chunks.append(group.iloc[::stride])
    result = pd.concat(chunks, ignore_index=True)
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def bounded_rows(frame, *, preferred=5000):
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    stride = max(1, math.ceil(len(frame) / preferred))
    result = frame.iloc[::stride].head(10000).copy()
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def snapshot_frame(snapshot, name):
    if name not in snapshot.datasets:
        return pd.DataFrame()
    return snapshot.frame(name)

def empty_panel(message):
    display(HTML(f'<div style="padding:18px;border:1px solid #d0d5dd;border-radius:8px;color:#475467">{message}</div>'))

In [ ]:
strategy_id = None
snapshot = execution_evidence(strategy_id, Window.days(30), adapter=adapter)
display(snapshot.provenance())
tca = snapshot_frame(snapshot, 'tca')
ledger = snapshot_frame(snapshot, 'runtime_ledger')
ledger_state_rows = snapshot.datasets.get('ledger_state', ())
ledger_state = ledger_state_rows[0] if ledger_state_rows else {
    'current_profitability_proven': False,
    'message': 'CURRENT PROFITABILITY UNPROVEN — runtime-ledger evidence is unavailable.',
    'historical_ledger_rows': 0,
}
if not ledger_state['current_profitability_proven']:
    display(HTML(f'''<div style="padding:18px;border:3px solid #b42318;border-radius:8px;
      font-size:20px;font-weight:800;background:#fff1f0">{ledger_state['message']}</div>'''))

## Slippage and shortfall distributions

In [ ]:
if tca.empty:
    empty_panel('No TCA rows in the selected window.')
else:
    numeric = ['slippage_bps', 'expected_shortfall_bps_p50', 'expected_shortfall_bps_p95',
               'realized_shortfall_bps', 'divergence_bps', 'filled_qty']
    tca[numeric] = tca[numeric].apply(pd.to_numeric, errors='coerce')
    tca_plot = bounded_rows(tca)
    fig = px.histogram(tca_plot, x='slippage_bps', color='symbol', marginal='box', nbins=50,
                       title='Slippage distribution by symbol', labels={'slippage_bps': 'slippage (bps)'})
    fig.show()
    scatter = tca_plot.dropna(subset=['expected_shortfall_bps_p50', 'realized_shortfall_bps']).copy()
    if scatter.empty:
        empty_panel('Expected versus realized shortfall has no paired observations.')
    else:
        fig = px.scatter(scatter, x='expected_shortfall_bps_p50', y='realized_shortfall_bps',
                         color='symbol', symbol='side', hover_data=['strategy_id', 'execution_id'],
                         title='Realized versus expected p50 shortfall')
        diagonal_max = max(scatter.expected_shortfall_bps_p50.max(), scatter.realized_shortfall_bps.max())
        diagonal_min = min(scatter.expected_shortfall_bps_p50.min(), scatter.realized_shortfall_bps.min())
        fig.add_shape(type='line', x0=diagonal_min, y0=diagonal_min, x1=diagonal_max, y1=diagonal_max,
                      line={'dash': 'dash', 'color': '#667085'})
        fig.show()
    fig = px.box(tca_plot, x='symbol', y='divergence_bps', color='side', points='outliers',
                 title='Shortfall divergence by symbol and side')
    fig.show()
    fig = px.box(tca_plot, x='strategy_id', y='slippage_bps', color='symbol', points='outliers',
                 title='Slippage distribution by strategy and symbol')
    fig.update_xaxes(tickangle=-35)
    fig.show()
    tca['has_fill'] = tca['filled_qty'].fillna(0) > 0
    coverage = tca.groupby(['strategy_id', 'symbol'], dropna=False).agg(
        tca_rows=('execution_id', 'count'), filled_rows=('has_fill', 'sum'),
        fill_coverage_ratio=('has_fill', 'mean'), filled_qty=('filled_qty', 'sum'),
        latest_tca=('computed_at', 'max')).reset_index()
    display(coverage.sort_values('tca_rows', ascending=False))

## Historical runtime-ledger evidence — not broker equity

In [ ]:
if ledger.empty:
    empty_panel('CURRENT PROFITABILITY UNPROVEN — no runtime-ledger rows exist in the bounded 180-day window.')
else:
    ledger['bucket_day'] = pd.to_datetime(ledger['bucket_day'], utc=True)
    ledger['net_strategy_pnl_after_costs'] = pd.to_numeric(ledger['net_strategy_pnl_after_costs'])
    ledger['stage_account'] = ledger['observed_stage'].astype(str) + ' / ' + ledger['account_label'].astype(str)
    ledger_plot = ledger.groupby(['bucket_day', 'stage_account'], as_index=False)['net_strategy_pnl_after_costs'].sum()
    ledger_plot = bounded_points(ledger_plot, 'stage_account', time_column='bucket_day')
    fig = px.line(ledger_plot, x='bucket_day', y='net_strategy_pnl_after_costs', color='stage_account',
                  markers=True, title='Historical runtime-ledger net P&L by stage/account (not broker equity)',
                  labels={'bucket_day': 'UTC day', 'net_strategy_pnl_after_costs': 'net P&L after modeled costs'})
    fig.update_layout(hovermode='x unified')
    fig.show()
    display(ledger.groupby(['observed_stage', 'account_label'], as_index=False).agg(
        first_bucket=('bucket_day', 'min'), latest_bucket=('latest_bucket_ended_at', 'max'),
        fills=('fill_count', 'sum'), net_pnl=('net_strategy_pnl_after_costs', 'sum')))

## Takeaways

TCA coverage and historical ledger evidence diagnose execution quality. They do not establish current profitability, broker equity, or permission to increase capital.